# Phase 4: Geometric Normalization
### Slant Correction & Standardized 16×16 Resampling
---
### What is Phase 4 and why do we need it?
When different people write by hand:
1. **Handwriting Tilt (Slant):** Some people write with an upright style, others lean heavily to the right (cursive forward slant), and some lean left.
2. **Variable Character Sizes:** An 'm' or 'w' is wide, an 'l' or '1' is tall and thin, and some letters are physically written larger than others.

A classical shallow neural network cannot accept images of random sizes, nor should it get confused simply because someone wrote with a $15^\circ$ cursive tilt.

**Phase 4 solves this through 3 simple stages:**
- **Stage 1 (Dual-Centroid Slant Estimation):** Measures the natural tilt angle of the handwriting.
- **Stage 2 (Affine Shear Deskewing):** Shifts the top of the letter horizontally to make it straight and upright, without clipping any pen strokes.
- **Stage 3 (Aspect-Ratio Preserving 16×16 Resampling):** Rescales the character into a $14 \times 14$ active box and centers it neatly inside a fixed $16 \times 16$ canvas with values between $0.0$ and $1.0$.

In [ ]:
# Step 1: Import libraries and Phase 4 modules
import sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Add Model directory to Python path
module_path = Path("../Model").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

from ingestion import ingest_image
from preprocessing import preprocess_image
from segmentation import segment_document
from normalization import (
    estimate_slant_angle,
    deskew_character,
    resize_and_center_16x16,
    normalize_character,
    normalize_all_characters,
    NormalizationEngine,
)
import config

print("All Phase 4 normalization modules loaded successfully!")

---
## Step 2: Extract Real Character Crops (Phases 1, 2, and 3)
We load our sample document, preprocess it, and extract segmented candidate characters.

In [ ]:
# Load test sample photograph
sample_file = Path("../data/samples/IMG_20261021_212517.jpg")
if not sample_file.exists():
    sample_file = Path("../data/samples/sample_camera_capture_synth.png")

# Execute Phases 1, 2, and 3
_, ingested, _ = ingest_image(sample_file)
prep = preprocess_image(ingested)
seg_result = segment_document(prep.binary_mask, prep.gray_image)

raw_characters = seg_result.get_all_characters()
print(f"Extracted {len(raw_characters)} raw character crops ready for geometric normalization.")

---
## Step 3: Understanding the Dual-Centroid Slant Method
How does the computer measure the slant angle $\theta$ of a letter?

1. We split the character vertically into two halves: **Top Zone** and **Bottom Zone**.
2. We calculate the center of mass (spatial centroid) of each zone: $(x_{\text{top}}, y_{\text{top}})$ and $(x_{\text{bot}}, y_{\text{bot}})$.
3. We calculate the horizontal and vertical offset:
   $$\Delta x = x_{\text{top}} - x_{\text{bot}}, \quad \Delta y = y_{\text{bot}} - y_{\text{top}}$$
4. The slant angle is simply:
   $$\theta = \arctan\left( \frac{\Delta x}{\Delta y} \right)$$

- If $\theta > 0$: The handwriting slants forward to the right.
- If $\theta < 0$: The handwriting slants backward to the left.
- If $\theta \approx 0$: The handwriting is upright.

In [ ]:
# Pick a sample character crop
sample_crop = raw_characters[12].binary_crop
slant_angle = estimate_slant_angle(sample_crop)

print(f"Sample Character Dimensions : {sample_crop.shape[1]}x{sample_crop.shape[0]} px")
print(f"Estimated Slant Angle       : {slant_angle:.2f} degrees")

---
## Step 4: Affine Horizontal Shear Deskewing
Once we know the slant angle $\theta$, we straighten it using **Affine Shear**:
- Shear factor: $s = \tan(\theta)$
- To ensure no ink strokes get cut off at the edge, we dynamically pad the image horizontally by $\lceil H \cdot |s| \rceil$ pixels.
- Then we apply `cv2.warpAffine` to make the stroke upright!

In [ ]:
# Demonstrate deskewing on a synthetic tilted stroke for crystal clear visualization
tilted_demo = np.zeros((50, 50), dtype=np.uint8)
cv2.line(tilted_demo, (15, 42), (32, 8), 255, 3)

demo_angle = estimate_slant_angle(tilted_demo)
tight_deskewed, full_sheared = deskew_character(tilted_demo, demo_angle)
remaining_angle = estimate_slant_angle(tight_deskewed)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(tilted_demo, cmap="gray")
axes[0].set_title(f"Before: Slanted ({demo_angle:.1f}°)", fontweight="bold")
axes[0].axis("off")

axes[1].imshow(tight_deskewed, cmap="gray")
axes[1].set_title(f"After Deskewing ({remaining_angle:.1f}°)", fontweight="bold")
axes[1].axis("off")

plt.tight_layout()
plt.show()

---
## Step 5: Aspect-Ratio Preserving 16×16 Resampling
*Why not just do `cv2.resize(crop, (16, 16))` directly?*
- Direct stretching destroys the shape of characters! A tall, thin '1' or 'l' would be stretched into a fat square block, and a wide 'w' would be crushed.

**Our Solution:**
1. Fit the character into an active inner $14 \times 14$ window while **preserving its exact aspect ratio**.
2. Symmetrically center it inside a fixed $16 \times 16$ black canvas.
3. Normalize all pixel brightness values to the range $[0.0, 1.0]$.

In [ ]:
# Normalize sample character
norm_char = normalize_character(sample_crop)

print(f"Normalized Image Shape : {norm_char.shape}")
print(f"Data Type              : {norm_char.normalized_image.dtype}")
print(f"Pixel Range            : [{norm_char.normalized_image.min():.2f}, {norm_char.normalized_image.max():.2f}]")

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(sample_crop, cmap="gray")
axes[0].set_title(f"1. Raw Crop ({sample_crop.shape[1]}x{sample_crop.shape[0]})")
axes[0].axis("off")

axes[1].imshow(norm_char.deskewed_crop, cmap="gray")
axes[1].set_title("2. Deskewed Crop")
axes[1].axis("off")

axes[2].imshow(norm_char.normalized_image, cmap="gray")
axes[2].set_title("3. Standard 16x16 Canvas")
axes[2].axis("off")

plt.tight_layout()
plt.show()

---
## Step 6: Normalizing the Full Batch of Extracted Characters
Let us normalize all the extracted characters from our handwritten page and inspect a grid of them.

In [ ]:
# Normalize all characters
engine = NormalizationEngine()
norm_result = engine.normalize_batch(raw_characters)

print(f"Total Characters Normalized : {norm_result.total_characters}")
print(f"Mean Handwriting Slant      : {norm_result.mean_slant_deg:.2f}°")

# Display first 24 normalized characters
fig, axes = plt.subplots(3, 8, figsize=(16, 6))
for i in range(24):
    ax = axes[i // 8, i % 8]
    ax.imshow(norm_result.characters[i].normalized_image, cmap="gray")
    ax.set_title(f"#{i+1}: {norm_result.characters[i].slant_angle_deg:.1f}°", fontsize=9)
    ax.axis("off")

plt.suptitle("Batch of Normalized 16x16 Characters (Ready for Phase 5 Feature Extraction)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Quick Professor & Viva Cheat Sheet (Phase 4)

> **Q1: What is the main objective of Phase 4?**  
> *Answer:* To remove handwriting slant variations and scale every isolated character crop into a standardized $16 \times 16$ float32 representation with preserved aspect ratio.
>
> **Q2: How does the Dual-Centroid slant estimation work?**  
> *Answer:* It divides the character vertically into top and bottom halves, computes the center of mass for each half, and finds the angle $\theta = \arctan(\Delta x / \Delta y)$ between the two centroids.
>
> **Q3: How do you prevent stroke clipping when shearing?**  
> *Answer:* Before applying the affine shear transformation, we add horizontal zero-padding equal to $\lceil H \cdot |\tan(\theta)| \rceil$ pixels on both sides so all sheared stroke pixels remain inside the canvas.
>
> **Q4: Why don't you directly resize characters to 16×16?**  
> *Answer:* Direct resizing distorts character aspect ratio (e.g. turning narrow digits like '1' into wide squares). Instead, we scale into a $14 \times 14$ active box maintaining aspect ratio and symmetrically pad with zeros.
>
> **Q5: Why is the output normalized to float32 in range [0.0, 1.0]?**  
> *Answer:* Neural networks converge faster and avoid vanishing/exploding gradients when numerical inputs are bounded in $[0.0, 1.0]$ rather than raw uint8 integers $[0, 255]$.